# Variational AutoEncoder (VAE)

## VAE

### prompt
```prompt
VAEとは？
```

### answer

Variational Autoencoder（VAE, 変分オートエンコーダ）は、**生成モデルとしても使える強化版Autoencoder**です。通常のAutoencoderに**確率的な解釈と生成能力**を加えたもので、2013年にKingmaとWellingによって提案されました。

---

🌱 **基本の考え方：確率モデルとしてのAutoencoder**

VAEの核心は、「データを潜在変数 $z$ によって生成されたもの」と考える**生成的確率モデル**です。

* 通常のAutoencoderは $z = f(x)$ のように**決定論的な潜在変数**を得ます。
* VAEでは $z \sim q(z \mid x)$ として、**確率的な潜在変数**を扱います。
* そこから再び $x \sim p(x \mid z)$ としてデータを**再構成・生成**します。

つまり、**入力データ → 分布（潜在表現） → 復元 or 新たな生成**という流れです。


📜 **ネットワーク構造**

VAEは以下の構造を持ちます：

---

### 1. Encoder（推論モデル）

* 入力 $x$ から**平均ベクトル** $\mu(x)$ と **分散ベクトル** $\sigma^2(x)$ を出力
* それから潜在変数 $z$ をサンプリング：

  $$z = \mu(x) + \sigma(x) \odot \epsilon,\quad \epsilon \sim \mathcal{N}(0, I)$$

  → **再パラメータ化トリック**で微分可能に

---

### 2. Decoder（生成モデル）

* サンプリングした $z$ から、元のデータ $x$ を復元・生成するネットワーク


**（おそらく、簡単すぎて分からない。）**

## 説明生成

コード生成と説明はAPIでgpt4-oを用いた。
一部、o3-mini-high説明を行った。

コードは一部修正しています。

### prompt
```prompt
variational autoencoderを説明してください。
autoencoderと比較して説明してください。
数式を用いてください。
```

![](fig/8200-vae-explain1.png)

![](fig/8200-vae-explain2.jpg)

![](fig/8200-vae-explain3.png)


### 注意
- $\mathbb{E}_{𝑞_𝜙 (𝑧|𝑥)} \left[ \log⁡𝑝_𝜃 (𝑥|𝑧) \right] $
は、$ 𝑞_𝜙 (𝑧|𝑥) $の分布で$\log⁡𝑝_𝜃 (𝑥|𝑧)$評価するという意味。

- 第一項と第二項の比率は場合により異なる。

### まとめ
上の1,2,3の数字に合わせて、NNアーキテクチャと損失関数は以下の通り。
![](fig/8200-vae-explain4.jpg)


### Autoencoder vs VAE

Autoencoder と VAE　の違い。

VAEのencoderは
1. 確率分布を出力する。
2. 確率分布から値を一つ取り出す。

以下は二次元のlatent spaceの例
![](fig/8200-vae-vs-ae.jpg)

## サンプリングと再パラメタ化

### prompt
```prompt
サンプリングと再パラメータ化について教えて下さい。
```

### answer

VAEの目的の一つは、入力データ $x$ に対する潜在変数 $z$ の確率分布を学習することです。この過程で、エンコーダーは入力データから潜在変数の平均 $\mu$ と分散 $\sigma^2$ を推定し、これらを用いて潜在変数 $z$ をサンプリングします。この潜在変数は通常、正規分布に従うと仮定されます。

---

## 再パラメータ化トリック

再パラメータ化トリックは、VAEを効果的にトレーニングするための技術です。これは、サンプリングによって生じる微分不可能な問題を解決します。

1. **課題**: 潜在変数 $z$ をエンコーダーが出力する平均 $\mu$ と分散 $\sigma^2$ を用いて正規分布から直接サンプリングすると、この操作は微分不可能になり、誤差逆伝播法（バックプロパゲーション）によってモデルを訓練することが難しくなります。
2. **解決策**: 再パラメータ化トリックを使って、サンプリングの不確実性を微分可能な形に変換します。

---

## 再パラメータ化トリックの実装

再パラメータ化トリックでは、以下のように潜在変数をサンプリングします：

* エンコーダーは、データ $x$ から $\mu$ と $\log(\sigma^2)$ を出力します。

* 標準正規分布からノイズ $\epsilon \sim \mathcal{N}(0, 1)$ をサンプリングします。

* 新しい潜在変数 $z$ を以下のように計算します：

  $$z = \mu + \sigma \cdot \epsilon$$

* ここで、$\sigma$ は標準偏差で、$\log(\sigma^2)$ の指数関数によって計算されます（つまり
  $\sigma = \exp\bigl(\log(\sigma^2) / 2\bigr)$
  ）。

この手法により、ランダムなサンプリングが $\mu$ と $\sigma$ に依存する形に転換され、微分可能なネットワーク構造となります。これにより、バックプロパゲーションを用いてVAEをトレーニングできるようになります。


### Q. なぜ、こんな面倒な手順を踏む？

![](fig/8200-vae-model-def.png)


# 再パラメタ化トリック

![](fig/8200-vae-reparametrization-trick.jpg)

## 再構成誤差

### prompt
```prompt
Variational antoencoderの再構成誤差を詳しく説明してください。
```
![](fig/8200-vae-reparametrization-trick-loss.jpg )


画像は規格化すると0-1なので誤差関数はどちらでも良い。
（AutoencoderのコードもBCEで計算可能→どちらが良いか？実行して評価する。）

## バイナリー交差エントロピー（BCE)

初めに，  

$x, \hat{x}$ : 入力データと生成された値  

- $x = 1 \rightarrow L = -\log(\hat{x})$  
- $x = 0 \rightarrow L = -\log(1 - \hat{x})$  

---

拡張：  
$x$ の 0–1 の間をつなぐと，二値交差エントロピー（BCE）は

$$
\mathrm{BCE}
  = -\bigl[\,x \log(\hat{x}) + (1 - x)\log(1 - \hat{x})\,\bigr]
$$

(実際は、$\log(x)$が発散しないように、$x, \hat{x}$を$ [ \delta, 1-\delta ] $の範囲内にする。$\delta$は微少量。)


MAEは全体を比較する。一方、BCEは
1ピクセルごと比較して平均。

![](fig/8200-vae-L-BCE.jpg)

## KL項
o3-mini-highで

### prompt
```prompt
VAEのKL divergence項 の式変形をを説明して下さい。

```
とした回答の要約を以下に表示する。


### answer
KLダイバージェンス（Kullback-Leibler divergence、カルバック・ライブラー情報量)：

#### 一般の定義
連続値、
$$
D_{KL}(P\|Q) = \sum_x P(x)\log\frac{P(x)}{Q(x)}
$$
離散値、
$$
D_{KL}(P\|Q) = \int dx\, P(x)\log\frac{P(x)}{Q(x)}
$$
は、𝑃(𝑥)（"正解データの分布" ）
に従うデータが与えられたとき、
そのデータを
 𝑄(𝑥)（"モデルが学習した分布"）
で表現したときの「情報のロス」

#### VAEの場合

正規分布 $P(𝑧)=𝑁(0,𝐼)$
に従うデータが与えられたとき、
そのデータを
VAEの潜在変数 𝑧の***全体の分布***
 $𝑄(𝑧∣𝑥) =𝑁(𝜇,𝜎^2)$
で表現したときの「情報のロス」

(個々分布（$𝜇_𝑖, 𝜎_𝑖^2$）正規分布に従うのではない。)


以下の定義
$$
Q(z) = \frac{1}{\sqrt{2\pi \sigma^2}} \exp\left(-\frac{(z - \mu)^2}{2\sigma^2}\right)
$$

$$
P(z) = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{z^2}{2}\right)
$$


を代入する。

$$
𝐷_{𝐾𝐿} (P||Q)=\int 𝑑𝑧 𝑃(𝑧) 𝑙𝑜𝑔 (𝑃(𝑧))/(𝑄(𝑧))
$$
$$
𝐷_{𝐾𝐿} (P||Q)
=−1/2  [(1+ log⁡(𝜎^2 )−𝜇^2−𝜎^2)]
$$




なお、この関数は$𝜇 =0, 𝜎^2=1$が最小値です。

上の説明は要約です。o3-mini-highは上の加えてかなり詳細に式変形して説明します。




### コード生成

#### prompt

```prompt
# 依頼
{#コード}をvariational antoencoderを用いたコードに書き換えてださい。

# コード
（autoencoderコードをコピペ。）
```

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from sklearn.datasets import load_digits
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import numpy as np
import random

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

# データをロードします
digits = load_digits()
data = digits.data
targets = digits.target

# データを正規化します
data = data / 16.0  # 0-16までの整数値なので16で割って正規化

# NumPy配列をPyTorchのテンソルに変換します
data = torch.tensor(data, dtype=torch.float32)
targets = torch.tensor(targets, dtype=torch.int64)

# TensorDatasetとDataLoaderを構築
dataset = TensorDataset(data, targets)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

class VAE(nn.Module):
    def __init__(self, input_dim, hidden_dim1, hidden_dim2, latent_dim):
        super(VAE, self).__init__()
        # エンコーダー: 入力層 -> 隠れ層1 -> 隠れ層2 -> 平均と標準偏差の線形層
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim1),
            nn.ReLU(),
            nn.Linear(hidden_dim1, hidden_dim2),
            nn.ReLU()
        )
        
        self.fc_mu = nn.Linear(hidden_dim2, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim2, latent_dim)
        
        # デコーダー: 潜在空間 -> 隠れ層1 -> 隠れ層2 -> 出力層
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim2),
            nn.ReLU(),
            nn.Linear(hidden_dim2, hidden_dim1),
            nn.ReLU(),
            nn.Linear(hidden_dim1, input_dim),
            nn.Sigmoid()
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

def loss_function(recon_x, x, mu, logvar):
    # バイナリ クロスエントロピ 損失と KL ダイバージェンス
    BCE = nn.functional.binary_cross_entropy(recon_x, x, reduction='sum')
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return BCE + KLD
    
# モデルの設定
input_dim = data.shape[1]  # 64 (8x8画像)
hidden_dim1 = 128
hidden_dim2 = 64
latent_dim = 2

# モデル、損失関数、最適化手法の設定
model = VAE(input_dim, hidden_dim1, hidden_dim2, latent_dim)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 学習を行います
num_epochs = 100
losses = []

for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    for batch_data, _ in dataloader:
        optimizer.zero_grad()
        reconstructed, mu, logvar = model(batch_data)
        loss = loss_function(reconstructed, batch_data, mu, logvar)
        loss.backward()
        train_loss += loss.item()
        optimizer.step()

    average_loss = train_loss / len(dataloader.dataset)
    
    losses.append(average_loss)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {average_loss:.4f}')

# 損失のプロット
plt.figure(figsize=(10, 5))
plt.plot(range(1, num_epochs + 1), losses, label='Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss per Epoch')
plt.legend()
plt.grid(True)
plt.show()

# 全データを用いて潜在変数を取得してプロット
model.eval()
with torch.no_grad():
    mu, logvar = model.encode(data)
    latent_space = model.reparameterize(mu, logvar).numpy()

plt.figure(figsize=(8, 6))
plt.scatter(latent_space[:, 0], latent_space[:, 1], alpha=0.5, c=targets, cmap='tab10')
plt.colorbar()
plt.title('Latent Space')
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.show()

上図は二次元潜在空間。𝑁(0,𝐼)に従うはず。


In [ ]:
# モデルを評価モードに変更
model.eval()
with torch.no_grad():
    # 先頭10件の訓練データを選択（X_trainはすでにTensorとして定義済み）
    sample_data = data[:10]
    # エンコーダを通して mu と logvar を取得
    mu, logvar = model.encode(sample_data)
    # sigma^2 を計算（σ^2 = exp(log σ^2)）
    sigma_squared = torch.exp(logvar)
    
    print("Selected training samples sigma^2:")
    print(sigma_squared)


In [ ]:
import matplotlib.pyplot as plt

# 訓練済みモデルを評価モードに切り替え
model.eval()

# 入力データをモデルに通して再構成を取得
with torch.no_grad():
    reconstructed_data, _ , _ = model(data)

# Tensor を numpy array に変換
input_data_np = data.numpy()
reconstructed_data_np = reconstructed_data.numpy()

# 比較するための画像数を設定
num_images_to_show = 10  # 例えば最初の10個を表示

# 図示
fig, axes = plt.subplots(nrows=2, ncols=num_images_to_show, figsize=(15, 4))
for i in range(num_images_to_show):
    # 入力画像
    axes[0, i].imshow(input_data_np[i].reshape(8, 8), cmap='gray')
    axes[0, i].axis('off')
    
    # 再構成画像
    axes[1, i].imshow(reconstructed_data_np[i].reshape(8, 8), cmap='gray')
    axes[1, i].axis('off')

axes[0, 0].set_title("Original", fontsize=12)
axes[1, 0].set_title("Reconstructed", fontsize=12)

plt.show()

二次元潜在空間を５０％拡張し、各軸１０分割して、対応する画像を並べてた。


In [ ]:
# ユーザー追加

factor = 0.5
(x1, y1), (x2, y2) = (latent_space.min(axis=0) - latent_space.ptp(axis=0)*factor,
                      latent_space.max(axis=0) + latent_space.ptp(axis=0)*factor)

print("xrange", x1,x2)
print("yrange", y1,y2)

In [ ]:
import numpy as np

# 潜在空間を10x10のグリッドに分割
x_range = (x1, x2)
y_range = (y2, y1)

# 分割数
n_bins = 10

# 潜在空間を10x10のグリッドに分割
fig, axarr = plt.subplots(n_bins, n_bins, figsize=(15, 15))

x_bins = np.linspace(x_range[0], x_range[1], n_bins)
y_bins = np.linspace(y_range[0], y_range[1], n_bins)

# 各グリッドに対応する潜在コードからデコードされた各画像を表示
for i, x_val in enumerate(x_bins):
    for j, y_val in enumerate(y_bins):
        # 選ばれた潜在変数のグリッドでの座標
        z = torch.tensor([[x_val, y_val]], dtype=torch.float32)

        # デコードして画像を生成
        with torch.no_grad():
            generated = model.decoder(z).numpy().reshape(8, 8)

        # 画像をプロット
        axarr[j, i].imshow(generated, cmap='gray')
        axarr[j, i].axis('off')

plt.suptitle('Decoding the Latent Space Grid', fontsize=16)
plt.subplots_adjust(wspace=0.1, hspace=0.1)
plt.show()

## VAEコードの説明
### prompt
```prompt
VAEのモデル定義と
損失関数の定義を詳しく説明してください。
コードと数式を関連付けてください。
```

### answer

#### model定義部分
![](fig/8200-vae-code-explain1.jpg)

補足：「対数分散にするのはなぜ？」と聞くと、「$𝜎^2$を評価するより計算が安定するから。」と回答する。$𝜎^2  = \exp⁡(\log⁡(𝜎^2))$


#### reparamatrization trick部分
![](fig/8200-vae-code-explain2.png)

実際のネットワークは$\mu$と$\log(\sigma^2)$を予想する。(すでに説明があったとおり、その方が安定するから。)
![](fig/8200-vae-reparametrization-trick-logsigma2.jpg)


![](fig/8200-vae-code-explain3.jpg)